# Solution: Discriminant models

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import torch

from cycler import cycler
import seaborn as sns

sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

## Introduction
In the lecture we've discussed how discriminant models are simple models that can be used to classify linearly separable data. 
However, even for linearly seperable data, discriminant models don't always behave as we would want, as we will see in the case of outliers.
To get a better understanding of the capabilities and limitations of discriminant models, we will implement the basic version and then extend them by adding basis functions.

In this notebook, we stick to a two-class classification problem as we did in the lecture, but return to a one-dimensional input space. 
Let's start by creating a basic dataset:  

In [ ]:
classes = 2
markers = ["o", "^"]
nums = [1, -1]
label_names = ["class 1", "class 2"]

# 1D target data [[x0, t0], [x1, t1], .. ]
data = torch.tensor(
    ([0.1, 1], [0.2, 1], [0.3, 1], [0.4, 1], [0.5, -1], [0.6, -1], [0.7, -1], [0.8, -1])
)

# Plot data
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

for i in range(classes):
    ax.plot(
        data[data[:, 1] == nums[i]][:, 0],
        data[data[:, 1] == nums[i]][:, 1],
        markers[i],
        c=colors[i],
        fillstyle="none",
        label=label_names[i],
    )
plt.legend()
plt.show()

It is easy to determine where the boundary between the classes should be in this example.
Let's try to solve this classification problem by minimizing a squared loss function.
This leads to a standard least squares problem.
Its solution is given by

$$
\bar{\mathbf{w}} = \left(\mathbf{X}^T \mathbf{X} \right)^{-1} \mathbf{X}^T \mathbf{t}.
$$

We can then compute our discriminant function $y(\mathbf{x})$ from the weights $\bar{\mathbf{w}}$:

$$
y(\mathbf{x},\bar{\mathbf{w}}) = \sum_{j=0}^M \bar{w}_j x_j = \bar{\mathbf{w}}^T  \mathbf{x}
$$

This discriminant function is then used to classify our data:
we classify as $\mathcal{C}_1$ if $y(\mathbf{x}) > 0$ and classify as $\mathcal{C}_2$ if $y(\mathbf{x}) < 0$.
The line defined by $y(\mathbf{x}) = 0$ is called the decision boundary.

Task: Complete the following functions:
- `solve_least_squares`: Computes the least-squares weights based on the inputs $\mathbf{X}$ and targets $\mathbf{t}$
- `y_func`: Returns the discriminant function on parameters $\bar{\mathbf{w}}$ and prediction location $\mathbf{x}$

In [ ]:
def solve_least_squares(X, t):
    """
    Solve w based on X and t
    """

    # ---------------------- student exercise --------------------------------- #
    w = torch.inverse(X.T @ X) @ X.T @ t  # Bishop 4.16
    # w = torch.matmul(torch.matmul(torch.inverse(torch.matmul(X.T, X)), X.T),t)  # This is equivalent to the above
    # ---------------------- student exercise --------------------------------- #

    return w

In [ ]:
def y_func(w, X):
    """
    Return computed value based on parameters w and input point x
    """

    # ---------------------- student exercise --------------------------------- #
    y = w @ X
    # y = torch.matmul(w, X)  # Equivalent
    # ---------------------- student exercise --------------------------------- #

    return y

We are now ready to solve the classification problem. 
We add a bias term to our dataset and then solve the least-squares problem to find $\bar{\mathbf{w}}$.

In [ ]:
# Get the inputs (as 2D array) and targets from the data
x = data[:, 0]
t = data[:, 1]

# Add a bias term to the input data
X = torch.concat((torch.ones((x.shape[0], 1)), x.view(-1, 1)), dim=1)

# Solve the least-squares problem to obtain the weights
w = solve_least_squares(X, t)
print(f"w: {w}")

How can we interpret these values of $\bar{\mathbf{w}}$?

Task: 
- Complete the function below that finds the location of the decision boundary based on $\bar{\mathbf{w}}$. (Note: this function only has to work for the specific case where x is a scalar, and $\bar{\mathbf{w}}$ is thus size 2.)

In [ ]:
def decision_boundary(w):
    # ---------------------- student exercise --------------------------------- #
    # w_0 * 1 + w_1 * x = 0
    # x = -w_0 / w_1
    x_decision_boundary = -w[0] / w[1]
    # ---------------------- student exercise --------------------------------- #
    return x_decision_boundary


## Plot the different classified regions
Now we plot our data $(\mathbf{X}, \mathbf{t})$, the discriminant function, and the resulting decision boundary. We find the location of the decision boundary analytically, knowing that $y(\mathbf{x},\mathbf{w}) = 0$ at the decision boundary.

In [ ]:
# Creating a grid of test points
dx = 0.01
x_test_origin = torch.arange(-0.1, 1.125, dx)

# Adding bias term
x_test = torch.concat(
    (torch.ones(x_test_origin.shape[0]).view(-1, 1), x_test_origin.view(-1, 1)), dim=1
)

# Compute the output for each test point
outputs = torch.empty((x_test.shape[0]))
for i in range(x_test.shape[0]):
    outputs[i] = y_func(w, x_test[i])

In [ ]:
# Plot data and discriminant model output
def plot_decisionboundary(w, data, colors):
    # Compute sets for plotting
    lb = torch.min(data[:, 0]) - 0.1
    ub = torch.max(data[:, 0]) + 0.1
    N_steps = 1000
    x_test_origin = torch.linspace(lb, ub, N_steps)
    x_test = torch.concat(
        (torch.ones(x_test_origin.shape[0]).view(-1, 1), x_test_origin.view(-1, 1)),
        dim=1,
    )

    # Compute outputs
    outputs = torch.empty((x_test.shape[0]))
    for i in range(x_test.shape[0]):
        outputs[i] = y_func(w, x_test[i])
    boundary = decision_boundary(w)
    print(f"Decision boundary at: {boundary.item():.4f}")

    # Plotting the decision boundary, y and data
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))

    ax.plot(x_test_origin, outputs, c="k", label="y(x)")
    ax.axvline(boundary.item(), c="C9", label="Decision boundary")

    for i in range(classes):
        ax.plot(
            data[data[:, 1] == nums[i]][:, 0],
            data[data[:, 1] == nums[i]][:, 1],
            markers[i],
            c=colors[i],
            fillstyle="none",
            label=label_names[i],
        )

    plt.xlim(lb, ub)
    plt.ylim(-1.1, 1.1)
    plt.legend(loc="center left")
    plt.show()

In [ ]:
plot_decisionboundary(w, data, colors)

In this figure we see the data points, the model output $y(x)$, and the decision boundary at $y(x)=0$.
The decision boundary is located at $x=0.45$, as you can see from the output above the figure.
This result is very similar to what we've seen in the lecture.
Now, let's dive a bit deeper into how this model performs on various other cases.

### Exercise: Sensitivity of the least-squares solution
Add more data points to the datasets (keep them linearly separable for now!), and observe what happens to the decision boundary when:
- Adding an outlier with a very large or small $x$ value.
- Duplicating one of the data points many times.

In [ ]:
# Modify this dataset:
new_data = torch.tensor(
    ([0.1, 1], [0.2, 1], [0.3, 1], [0.4, 1], [0.5, -1], [0.6, -1], [0.7, -1], [0.8, -1])
)  # Default

# ---------------------- student exercise --------------------------------- #
new_data = torch.tensor(
    (
        [-8, 1],
        [0.1, 1],
        [0.2, 1],
        [0.3, 1],
        [0.4, 1],
        [0.5, -1],
        [0.6, -1],
        [0.7, -1],
        [0.8, -1],
    )
)  # Example outlier
# new_data = torch.tensor(([0.1, 1], [0.2, 1], [0.3, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.4, 1], [0.5,-1], [0.6,-1], [0.7,-1], [0.8,-1]))  # Example duplicates
# ---------------------- student exercise --------------------------------- #

In [ ]:
# Again get the inputs and targets and add a bias term
x = new_data[:, 0]
t = new_data[:, 1]
X = torch.concat((torch.ones((x.shape[0], 1)), x.view(-1, 1)), dim=1)

# Get the new least-squares weights, and plot the result
w = solve_least_squares(X, t)
plot_decisionboundary(w, new_data, colors)

Does the discriminant model correctly classify a linearly seperable dataset in a robust manner?
What is the underlying issue?

Later on in the course we will see models which can handle these cases a bit better.

## A more complex case 
As we have seen in the lecture, the least squares solution we used so far is not able to classify data that is not linearly separable.
We can use basis functions to add non-linear features to our model and attempt to classify more complex data.
We start by visualizing a more challenging test case:

In [ ]:
# Creating data for a more complex case that can't be seperated linearly
def true_func(x):
    y_r = torch.sin(2.5 * torch.pi * x + 1)
    y_c = [int(torch.floor(x) if x < 0 else torch.ceil(x)) for x in y_r]
    return torch.tensor(y_c)


x = torch.linspace(0, 1, 21)
t = true_func(x)
complex_data = torch.stack((x, t), dim=1)

# Plot data
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
for i in range(classes):
    ax.plot(
        complex_data[complex_data[:, 1] == nums[i]][:, 0],
        complex_data[complex_data[:, 1] == nums[i]][:, 1],
        markers[i],
        c=colors[i],
        fillstyle="none",
        label=label_names[i],
    )
plt.legend(loc="center left")
plt.show()

As you would expect, assuming a linear decision boundary with our current model leads to a bad result:

In [ ]:
# Get the inputs and targets and add a bias term
x = complex_data[:, 0]
t = complex_data[:, 1]
X = torch.concat((torch.ones((x.shape[0], 1)), x.view(-1, 1)), dim=1)

# Get the new least-squares weights, and plot the result
w = solve_least_squares(X, t)
plot_decisionboundary(w, complex_data, colors)

The computed decision boundary is so terrible that it's outside of the data region.

## Adding basis functions
Clearly, our model is not able to classify the data correctly. 
We can improve our model by adding our input space using basis functions, analogous to the regression case.

We obtain the linear model with nonlinear basis functions by replacing the coordinate vector $\mathbf{x}$ with the feature vector $\boldsymbol{\phi}( \mathbf{x} )$: 

$$
y(\mathbf{x},\mathbf{w}) = \sum_{j=0}^M w_j \phi_j(\mathbf{x}) = \mathbf{w}^T \boldsymbol{\phi} (\mathbf{x})
$$

We still use the least-squares function defined earlier to solve for $\bar{\mathbf{w}}$:

$$
\bar{\mathbf{w}} = \left(\mathbf{\Phi}^T \boldsymbol{\Phi} \right)^{-1} \boldsymbol{\Phi}^T \mathbf{t}
$$

### Exercise: basis function
Your task is to implement a radial basis function that expands a one-dimensional input to $M$ dimensions.
You will need to decide how to allocate the basis functions over the domain, initially we will stick to only having two basis functions to simplify plotting.

In [ ]:
# Here is a function for the RadialBasisFunctions:
def RadialBasisFunctions_1D(x_input, domain, M_radial, l_radial):
    """
    A function that computes radial basis functions.

    Arguments:
    x_input  -  The points for which to compute Phi
    domain   -  The domain [left boundary, right boundary] of the input data
    M_radial -  The number of basis functions
    l_radial -  The width of each basis function
    :return: -  Phi, which has shape (x_input.shape[0], M_radial)
    """

    # ---------------------- student exercise --------------------------------- #
    # Get locations of mu
    mus = torch.linspace(domain[0], domain[1], M_radial)

    # Compute Phi
    Phi = torch.zeros((x_input.shape[0], M_radial))
    for j, mu in enumerate(mus):
        Phi[:, j] = torch.exp(-((x_input - mu) ** 2) / (2 * l_radial**2))
    # ---------------------- student exercise --------------------------------- #

    return Phi

Now we solve the least-squares problem, and expand our test points to plot the resulting predictions. 
All the steps taken in this next codeblock should be clear to you:

In [ ]:
# Parameters for the basis functions
domain = [0, 1]
M_radial = 2
l_radial = 0.3

# Expand the data to the basis functions
Phi = RadialBasisFunctions_1D(x, domain, M_radial, l_radial)
# Add a bias term
Phi = torch.concat((torch.ones((Phi.shape[0], 1)), Phi), dim=1)

# Solving for w
w = solve_least_squares(Phi, t)

# Expand the test points x_test to phi_test through the basis function
Phi_test = RadialBasisFunctions_1D(x_test_origin, domain, M_radial, l_radial)
# Add a bias term
Phi_test = torch.concat((torch.ones((Phi_test.shape[0], 1)), Phi_test), dim=1)

# Compute the output for each test point
y_test = torch.zeros((x_test.shape[0]))
for i in range(x_test.shape[0]):
    y_test[i] = y_func(w, Phi_test[i])

Now we can plot our solution:

In [ ]:
# Plotting data
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

# Plot the classification regions
class_test = torch.tensor([0 if x > 0 else 1 for x in y_test])
for i in range(x_test.shape[0] - 1):
    ax.fill_between(
        [x_test_origin[i] - dx / 2, x_test_origin[i + 1] - dx / 2],
        -1.1,
        1.1,
        color=colors[class_test[i]],
        alpha=0.2,
        edgecolor="none",
    )

# Plot the classified data
for i in range(classes):
    x_data = complex_data[:, 0]
    t_data = complex_data[:, 1]
    include = t_data == nums[i]
    ax.plot(
        x_data[include],
        t_data[include],
        markers[i],
        c=colors[i],
        fillstyle="none",
        label=label_names[i],
    )

# Plot the discriminant function
ax.plot(x_test_origin, y_test, c="k", label="y(x)")

# Legend without duplicate labels
handles, labels = ax.get_legend_handles_labels()
handles.extend(
    [
        mpl.patches.Patch(facecolor=colors[i], edgecolor="k", alpha=0.3)
        for i in range(classes)
    ]
)
labels.extend([f"Classified to {label_names[i]}" for i in range(classes)])
plt.legend(handles=handles, labels=labels, fontsize=10, loc="center left")

plt.xlim(-0.1, 1.1)
plt.ylim(-1.1, 1.1)
plt.xlabel("x")
plt.ylabel("Class")
plt.show()

You should observe that the model with basis functions is much more capable in classifying the problem, as it is not limited by a single decision boundary in this space.

To demonstrate that least-squares is still finding a linear solution, we plot the output of the model in the basis function space:

In [ ]:
# Create grid in phi space, ignore this code
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

lower = min(torch.min(Phi[:, 1]), torch.min(Phi[:, 2])) - 0.1
upper = max(torch.max(Phi[:, 1]), torch.max(Phi[:, 2])) + 0.1
n_val = 100

phi_data = RadialBasisFunctions_1D(complex_data[:, 0], domain, M_radial, l_radial)

p1val = torch.linspace(lower, upper, n_val)
p2val = torch.linspace(lower, upper, n_val)

p1p1, p2p2 = torch.meshgrid(p1val, p2val, indexing="ij")
class_val = torch.empty((n_val, n_val))

for i in range(len(p1val)):
    for j in range(len(p2val)):
        x_pred = torch.tensor([1, p1val[i], p2val[j]])
        y_pred = y_func(w, x_pred)
        if y_pred >= 0:
            class_val[i, j] = 1
        else:
            class_val[i, j] = 2

ax.set_xlabel(r"First basis function $\phi_1(x)$")
ax.set_ylabel(r"Second basis function $\phi_2(x)$")
ax.contourf(p1p1, p2p2, class_val, alpha=0.3, colors=colors[:2], levels=1)
for i in range(classes):
    ax.plot(
        phi_data[complex_data[:, 1] == nums[i]][:, 0],
        phi_data[complex_data[:, 1] == nums[i]][:, 1],
        markers[i],
        c=colors[i],
        fillstyle="none",
        label=label_names[i],
    )

handles, labels = ax.get_legend_handles_labels()
handles.extend(
    [
        mpl.patches.Patch(facecolor=colors[i], edgecolor="k", alpha=0.3)
        for i in range(classes)
    ]
)
labels.extend([f"Classified to {label_names[i]}" for i in range(classes)])

ax.legend(handles=handles, labels=labels, fontsize=10)
plt.show()

In this figure, you should observe a linear decision boundary in the basis function space. 

## Review:
In this exercise, you've become familiar with using a discriminant model for classification. It is important to understand the capabilities and limitations of discriminant models. Change the data and get familiar with how the model classifies it, then answer the following questions:
- Why does the decision boundary change when adding duplicate points?
- What happens when we add outliers to this more complex case?
- If we use more basis functions, is the decision boundary when plotting the first two still linear?